# Q1. 996. Chunk-level retrieval





Chunk-level retrieval is a simple way to find the most relevant piece of a document for a query using embedding similarity. You’ll build a tiny retriever that embeds a query and many text chunks, then returns the top matches by cosine similarity.

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
⌄
⌄
import numpy as np

def chunk_level_retrieval(query_emb, chunk_embs, top_k):
    """
    Retrieves the most relevant chunks for a query using cosine similarity.

    Cosine similarity is:
        sim(q, c) = (q · c) / (||q|| * ||c||)

    Args:
        query_emb (np.ndarray): Query embedding of shape (d,).
        chunk_embs (np.ndarray): Chunk embeddings of shape (n_chunks, d).
        top_k (int): Number of chunks to return.

    Returns:
        np.ndarray: Indices of the top_k most similar chunks, sorted by decreasing similarity.
    """
    # Your solution below
Rules:

Compute cosine similarity between query_emb and every chunk embedding in chunk_embs.
Return the indices of the top_k chunks with the highest similarity, sorted best-to-worst.
Break ties by returning the smaller index first.
Don’t use any prebuilt similarity/search utilities (e.g., sklearn, faiss).
Example
python
Copy
1
2
3
4
5
6
7
8
9
⌄
query_emb = [1.0, 0.0]
chunk_embs = [
    [1.0, 0.0],   # very similar
    [0.0, 1.0],   # not similar
    [0.7, 0.7]    # somewhat similar
]
top_k = 2

chunk_level_retrieval(query_emb, chunk_embs, top_k)
Output:

python
Copy
1
[0, 2]


In [ ]:
import numpy as np


def chunk_level_retrieval(query_emb, chunk_embs, top_k):
    """
    Retrieves the most relevant chunks for a query using cosine similarity.

    Cosine similarity is:
        sim(q, c) = (q · c) / (||q|| * ||c||)

    Args:
        query_emb (np.ndarray): Query embedding of shape (d,).
        chunk_embs (np.ndarray): Chunk embeddings of shape (n_chunks, d).
        top_k (int): Number of chunks to return.

    Returns:
        np.ndarray: Indices of the top_k most similar chunks, sorted by
        decreasing similarity.
    """
    # Convert inputs to NumPy arrays (safeguard)
    query_emb = np.asarray(query_emb)
    chunk_embs = np.asarray(chunk_embs)

    # Compute dot products between query and all chunks
    dots = chunk_embs @ query_emb

    # Compute norms for cosine similarity
    query_norm = np.linalg.norm(query_emb)
    chunk_norms = np.linalg.norm(chunk_embs, axis=1)

    # Compute cosine similarities
    sims = dots / (query_norm * chunk_norms)

    # Sort by descending similarity, then ascending index for ties
    indices = np.arange(len(chunk_embs))
    order = np.lexsort((indices, -sims))

    # Select top_k indices
    top_indices = order[:top_k]

    return top_indices

result = chunk_level_retrieval(query_emb, chunk_embs, top_k)
result

# Q2.. 725. Batched query retrieval




Implement batched query retrieval to find the top‑
k
k most similar items (by cosine similarity) in a small embedding index for each query. You’ll take queries and documents as NumPy arrays and return, for every query, the indices of the best matches.

The cosine similarity between vectors 
q
q and 
d
d is:

cos
⁡
(
q
,
d
)
=
q
⋅
d
∥
q
∥
2
 
∥
d
∥
2
cos(q,d)= 
∥q∥ 
2
​
 ∥d∥ 
2
​
 
q⋅d
​
 
Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
⌄
⌄
def batched_retrieve_topk(queries, docs, k):
    """
    Retrieves top-k document indices for each query using cosine similarity.

    Args:
        queries (np.ndarray): Query embeddings of shape (num_queries, dim).
        docs (np.ndarray): Document embeddings of shape (num_docs, dim).
        k (int): Number of top documents to retrieve per query.

    Returns:
        np.ndarray: For each query, a list of length k containing the
        document indices (0-based) with highest cosine similarity, sorted
        from most similar to least similar. Shape (num_queries, k).
    """
    # Your solution below
Rules:

Compute cosine similarities between every query and every document (batched).
Return only the document indices, not the similarity scores.
Break ties by returning smaller document indices first.
Don’t use any prebuilt retrieval or similarity search libraries (e.g., FAISS, sklearn neighbors).
Use NumPy operations for the main computation (avoid Python double for-loops over queries×docs).
Return a NumPy array.
Example
python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
⌄
⌄
queries = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
])
docs = np.array([
    [1.0, 0.0],   # idx 0
    [0.9, 0.1],   # idx 1
    [0.0, 1.0],   # idx 2
])
k = 2

batched_retrieve_topk(queries, docs, k)
Output:

python
Copy
1
2
3
4
⌄
array([
    [0, 1],
    [2, 1]
])


In [ ]:
import numpy as np

def batched_retrieve_topk(queries, docs, k):
    """
    Retrieves top-k document indices for each query using cosine similarity.

    Args:
        queries (np.ndarray): Query embeddings of shape (num_queries, dim).
        docs (np.ndarray): Document embeddings of shape (num_docs, dim).
        k (int): Number of top documents to retrieve per query.

    Returns:
        np.ndarray: For each query, a list of length k containing the
        document indices (0-based) with highest cosine similarity, sorted
        from most similar to least similar. Shape (num_queries, k).
    """
    nq, nd = queries.shape[0], docs.shape[0]
    k = max(0, min(int(k), nd))

    # Precompute norms
    # Use a small epsilon to avoid division by zero, but use maximum to avoid
    # distorting correlations for valid vectors (unlike adding eps).
    eps = 1e-12
    q_norms = np.linalg.norm(queries, axis=1)
    d_norms = np.linalg.norm(docs, axis=1)

    q_norms = np.maximum(q_norms, eps)
    d_norms = np.maximum(d_norms, eps)

    # Compute cosine similarity matrix
    dots = queries @ docs.T                                    # (nq, nd)
    sims = dots / (q_norms[:, None] * d_norms[None, :])  # (nq, nd)

    # Stable sort indices by similarity descending
    # stable sort is needed for consistent tie-breaking by index
    order = np.argsort(-sims, axis=1, kind="mergesort")

    # Return top-k indices
    return order[:, :k]

result = batched_retrieve_topk(queries, docs, k)
result

# Q3. 373. Early stopping retrieval




Early stopping can save compute by stopping a similarity search once you’ve already found “good enough” matches for a query embedding. In this question, you’ll implement a simple early-stopping rule for cosine-similarity retrieval over a list of candidate embeddings.

The cosine similarity between a query vector 
q
q and a candidate vector 
c
c is defined as:

sim
(
q
,
c
)
=
q
⋅
c
∥
q
∥
∥
c
∥
sim(q,c)= 
∥q∥∥c∥
q⋅c
​
 
Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
⌄
⌄
def early_stop_retrieval(query, candidates, top_k, threshold, patience):
    """
    Retrieves top-k candidates by cosine similarity with early stopping.

    Args:
        query (list[float]): Query embedding of shape (d,).
        candidates (list[list[float]]): Candidate embeddings of shape (n, d).
        top_k (int): Number of results to return.
        threshold (float): Early-stop if best similarity >= threshold for `patience` consecutive checks.
        patience (int): Number of consecutive candidates meeting the threshold condition to stop.

    Returns:
        list[tuple[int, float]]: A list of (index, similarity) for the top_k matches found,
        sorted by descending similarity.
    """
    # Your solution below
Cosine similarity is:

cos
⁡
(
q
,
x
)
=
q
⋅
x
∥
q
∥
∥
x
∥
cos(q,x)= 
∥q∥∥x∥
q⋅x
​
 
Rules:

Compute cosine similarity between query and each candidate in the given order.
Maintain the current top_k results (indices + scores) as you scan.
Early stop when the current best similarity seen so far is >= threshold for patience consecutive candidate checks.
Return the best top_k matches found up to the point you stop (or after scanning all candidates).
Do not use any prebuilt nearest-neighbor or retrieval utilities (e.g., FAISS, sklearn neighbors).
Example
python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
⌄
query = [1.0, 0.0]
candidates = [
    [0.9, 0.1],  # cos ~ 0.9939
    [0.0, 1.0],  # cos = 0.0
    [1.0, 0.0],  # cos = 1.0
    [0.8, 0.2],  # cos ~ 0.9701
]
top_k = 2
threshold = 0.99
patience = 2

early_stop_retrieval(query, candidates, top_k, threshold, patience)
Output:

python
Copy
1
[(2, 1.0), (0, 0.9939)]


In [ ]:
import numpy as np

def early_stop_retrieval(query, candidates, top_k, threshold, patience):
    """
    Retrieves top-k candidates by cosine similarity with early stopping.

    Args:
        query (np.ndarray): Query embedding of shape (d,).
        candidates (np.ndarray): Candidate embeddings of shape (n, d).
        top_k (int): Number of results to return.
        threshold (float): Early-stop if best similarity >= threshold for
            `patience` consecutive checks.
        patience (int): Number of consecutive candidates meeting the
            threshold condition to stop.

    Returns:
        list[tuple[int, float]]: A list of (index, similarity) for the top_k
        matches found, sorted by descending similarity.
    """

    # Compute query norm
    q_norm = np.linalg.norm(query)
    # Avoid division by zero if query is all zeros
    if q_norm == 0:
        return []

    # Track top-k as list of (idx, sim)
    top = []
    best_sim = float("-inf")
    streak = 0

    for idx in range(candidates.shape[0]):
        cand = candidates[idx]

        # Compute cosine similarity
        c_norm = np.linalg.norm(cand)
        if c_norm == 0:
            sim = 0.0
        else:
            sim = np.dot(query, cand) / (q_norm * c_norm)

        # Update top-k list
        if len(top) < top_k:
            top.append((idx, sim))
            top.sort(key=lambda x: x[1], reverse=True)
        else:
            if sim > top[-1][1]:
                top[-1] = (idx, sim)
                top.sort(key=lambda x: x[1], reverse=True)

        # Update best similarity and early-stop streak
        if sim > best_sim:
            best_sim = sim
        if best_sim >= threshold:
            streak += 1
        else:
            streak = 0

        if streak >= patience:
            break

    return top

result = early_stop_retrieval(query, candidates, top_k, threshold, patience)
result

# Q4. 1227. Embedding collapse detection




Detect embedding collapse in an embeddings-and-retrieval pipeline, where many items unintentionally map to nearly the same vector and retrieval quality degrades. You’ll compute a simple “collapse score” based on average cosine similarity across embedding pairs.

The cosine similarity between two vectors 
u
u and 
v
v is:

cos
⁡
(
u
,
v
)
=
u
⋅
v
∥
u
∥
∥
v
∥
cos(u,v)= 
∥u∥∥v∥
u⋅v
​
 
Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
⌄
⌄
def detect_embedding_collapse(embeddings, threshold=0.95):
    """
    Detects embedding collapse using average pairwise cosine similarity.

    Args:
        embeddings (np.ndarray): Item embeddings of shape (n_items, dim).
        threshold (float): If the average pairwise cosine similarity is >= threshold,
            report collapse.

    Returns:
        tuple[float, bool]:
            (avg_pairwise_cosine_similarity, is_collapsed)
    """
    # Your solution below
Rules:

Convert the input to a NumPy array if necessary and L2-normalize each embedding vector.
Compute the average cosine similarity over all unique pairs (i, j) where i < j.
Return the average similarity and a boolean is_collapsed based on threshold.
Don’t use any prebuilt similarity utilities (e.g., sklearn.metrics.pairwise).
Keep it in a single Python function using only NumPy (and Python built-ins if needed).
Example
python
Copy
1
2
3
4
5
6
7
⌄
embeddings = np.array([
    [1.0, 0.0],
    [0.99, 0.01],
    [0.98, 0.02]
])

detect_embedding_collapse(embeddings, threshold=0.95)
Output:

python
Copy
1
(0.9997, True)


In [ ]:
import numpy as np


def detect_embedding_collapse(embeddings, threshold=0.95):
    """
    Detects embedding collapse using average pairwise cosine similarity.

    Args:
        embeddings (np.ndarray): Item embeddings of shape (n_items, dim).
        threshold (float): If the average pairwise cosine similarity is >=
            threshold, report collapse.

    Returns:
        tuple[float, bool]:
            (avg_pairwise_cosine_similarity, is_collapsed)
    """

    # L2-normalize each embedding vector
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Avoid division by zero
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

    # Compute cosine similarity matrix
    sim_matrix = embeddings @ embeddings.T

    # Extract upper-triangular (i < j) similarities
    n_items = embeddings.shape[0]
    triu_idx = np.triu_indices(n_items, k=1)
    pair_sims = sim_matrix[triu_idx]

    # Average pairwise similarity
    avg_sim = float(pair_sims.mean()) if pair_sims.size > 0 else 1.0

    # Determine collapse
    is_collapsed = avg_sim >= threshold
    return avg_sim, is_collapsed

result = detect_embedding_collapse(embeddings, threshold)
result

# Q5. 1050. Negative sampling




Implement negative sampling for embedding-based retrieval, where you pick “hard negatives” that are most similar to a query but are not its positive match. You’ll compute cosine similarities and return the top-k negative candidate IDs for each query.

The cosine similarity is:

cos
⁡
(
q
,
d
)
=
q
⋅
d
∥
q
∥
2
 
∥
d
∥
2
cos(q,d)= 
∥q∥ 
2
​
 ∥d∥ 
2
​
 
q⋅d
​
 
Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
⌄
⌄
import numpy as np

def negative_sampling(queries, docs, positives, k):
    """
    Selects hard negatives for each query in an embedding retrieval setting.

    Args:
        queries (np.ndarray): Query embeddings of shape (n_queries, dim).
        docs (np.ndarray): Document/item embeddings of shape (n_docs, dim).
        positives (np.ndarray): Array of shape (n_queries,) where positives[i] 
            is the index of the positive doc for query i.
        k (int): Number of negatives to return per query.

    Returns:
        np.ndarray: A matrix of shape (n_queries, k) containing document indices
        representing negatives, chosen as the k most similar docs
        excluding the positive.
    """
    # Your solution below
Rules:

Use cosine similarity to score each (query, doc) pair.
For each query i, exclude positives[i] from being selected as a negative.
Return the k doc indices with highest similarity (hard negatives) for each query.
Do not use any prebuilt nearest-neighbor or retrieval libraries (e.g., FAISS, sklearn neighbors).
Key considerations: normalize embeddings for cosine; use vectorized NumPy ops where possible; rank by descending similarity.
Example
python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
⌄
⌄
queries = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
])
docs = np.array([
    [1.0, 0.0],   # doc 0
    [0.8, 0.2],   # doc 1
    [0.0, 1.0],   # doc 2
    [0.2, 0.8],   # doc 3
])
positives = np.array([0, 2])
k = 2

negative_sampling(queries, docs, positives, k)
Output:

python
Copy
1
2
3
4
⌄
np.array([
    [1, 3],  # for query 0, best negatives are docs 1 and 3 (excluding doc 0)
    [3, 1],  # for query 1, best negatives are docs 3 and 1 (excluding doc 2)
])


In [ ]:
import numpy as np


def negative_sampling(queries, docs, positives, k):
    """
    Selects hard negatives for each query in an embedding retrieval setting.

    Args:
        queries (np.ndarray): Query embeddings of shape (n_queries, dim).
        docs (np.ndarray): Document/item embeddings of shape (n_docs, dim).
        positives (np.ndarray): Array of shape (n_queries,) where positives[i] 
            is the index of the positive doc for query i.
        k (int): Number of negatives to return per query.

    Returns:
        np.ndarray: A matrix of shape (n_queries, k) containing document indices
        representing negatives, chosen as the k most similar docs
        excluding the positive.
    """
    # Normalize embeddings for cosine similarity
    q_norm = queries / np.linalg.norm(queries, axis=1, keepdims=True)
    d_norm = docs / np.linalg.norm(docs, axis=1, keepdims=True)

    # Compute cosine similarities
    sim = q_norm @ d_norm.T

    # Exclude positives by setting their similarity to -inf
    sim[np.arange(sim.shape[0]), positives] = -np.inf

    # Get top-k indices per query
    # argpartition gets the top k elements but not necessarily sorted
    topk_idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]

    # We need to sort these top k elements by their scores descending
    topk_scores = np.take_along_axis(sim, topk_idx, axis=1)
    order = np.argsort(-topk_scores, axis=1)
    topk_sorted = np.take_along_axis(topk_idx, order, axis=1)

    return topk_sorted

result = negative_sampling(queries, docs, positives, k)
result